# Week 7 — Merging DataFrames: Joins & pd.merge()
## Phase 2a Python | PORA Academy Cohort 7 — Wednesday Exercises

Welcome to your Wednesday practice notebook. Today you learned how to combine
separate tables into one using **joins** (inner, left, right, outer) and the
`pd.merge()` function. These exercises let you practise on the real Olist e-commerce data.

**How to work through this notebook:**
- **Read each question carefully** before writing any code.
- Run the **Data Setup** cell first — every question depends on it.
- Write your answer in the cell marked `# Your code here`.
- Each question shows the **expected output** so you can check yourself. If your
  numbers don't match, re-read the question — a wrong join type (`how=`) or a wrong
  key column (`on=`) is the usual culprit.
- Work through Questions 1–5 individually, then tackle the **Group Exercise**
  (Questions 6–8) with your team.

**Reminder — the four join types:**

| Join | What it keeps |
|---|---|
| `how='inner'` (default) | only rows that match in **both** tables |
| `how='left'` | **all** left rows + matched right rows (unmatched → `NaN`) |
| `how='right'` | **all** right rows + matched left rows (unmatched → `NaN`) |
| `how='outer'` | **all** rows from **both** tables (unmatched → `NaN`) |


In [1]:
# ── DATA SETUP — run this first ──────────────────────────────
# Standard Olist data loading for Google Colab (Week 3 onwards)
import pandas as pd
from google.colab import drive
import os

# Mount Google Drive
#drive.mount('/content/drive')

# Define path to the shared olist-data folder
olist_path = '/content/drive/MyDrive/olist-data'

# Load the tables you need this session
orders = pd.read_csv(os.path.join(olist_path, '/content/olist_orders_dataset.csv'))
customers = pd.read_csv(os.path.join(olist_path, '/content/olist_customers_dataset.csv'))
order_items = pd.read_csv(os.path.join(olist_path, '/content/olist_order_items_dataset.csv'))
products = pd.read_csv(os.path.join(olist_path, '/content/olist_products_dataset.csv'))
product_translations = pd.read_csv(os.path.join(olist_path, '/content/product_category_name_translation.csv'))

# Confirm everything loaded
print(f"✓ orders:              {orders.shape}")
print(f"✓ customers:           {customers.shape}")
print(f"✓ order_items:         {order_items.shape}")
print(f"✓ products:            {products.shape}")
print(f"✓ product_translations:{product_translations.shape}")

✓ orders:              (99441, 8)
✓ customers:           (99441, 5)
✓ order_items:         (112650, 7)
✓ products:            (32951, 9)
✓ product_translations:(71, 2)


## Question 1 — Inspect the two tables before joining

Before you merge anything, always check the shape of each table and confirm they
share a join key. Both `orders` and `customers` have a `customer_id` column.

Print the shape of `orders` and the shape of `customers`.

**Expected output:**
```
orders shape:    (99441, 8)
customers shape: (99441, 5)
```


In [9]:
orders.head(1)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00


In [10]:
customers.head(1)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP


In [11]:
order_items.head(1)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29


In [12]:
product_translations.head(1)

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty


In [13]:
products.head(1)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0


In [2]:
print('orders shape:', orders.shape)
print('customers shape:', customers.shape)


orders shape: (99441, 8)
customers shape: (99441, 5)


## Question 2 — Inner merge: orders + customers

Merge `orders` with `customers` on `customer_id` using an **inner** join (the
default). Store the result in a variable called `oc` and print its shape.

Because every `customer_id` in `orders` also exists in `customers`, no rows are
lost — but the column count grows as the customer columns are added on.

**Expected output:**
```
(99441, 12)
```


In [14]:
oc = orders.merge(customers, on='customer_id')  # inner join is the default how='inner'
print(oc.shape)

(99441, 12)


## Question 3 — Orders per state

Now that customer details are attached, you can analyse orders by location.
Using your merged `oc` table, group by `customer_state`, count the `order_id`
values, sort the result from most to fewest, and print the **top 5 states**.

**Expected output:**
```
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
Name: order_id, dtype: int64
```


In [15]:
top5_states = (
    oc.groupby('customer_state')['order_id']
      .count()
      .sort_values(ascending=False)
      .head(5)
)
print(top5_states)

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
Name: order_id, dtype: int64


## Question 4 — Left join: order_items + products (watch for unmatched rows)

You want to know the product **category** for every order item. Merge
`order_items` with just two columns of `products` —
`products[['product_id', 'product_category_name']]` — on `product_id`, using a
**left** join so that no order items are dropped even if the product is missing.

Store the result in `items_products`, then:
1. Print its shape.
2. Print how many rows have a **missing** `product_category_name` (use
   `.isnull().sum()`).

A left join keeps all 112,650 items; the missing values reveal items whose
product record has no category.

**Expected output:**
```
(112650, 8)
1603
```


In [16]:
items_products = order_items.merge(
    products[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
)

print(items_products.shape)
print(items_products['product_category_name'].isnull().sum())


(112650, 8)
1603


## Question 5 — Add English category names, then rank by revenue

The `product_category_name` values are in Portuguese. Left-join your
`items_products` table to `product_translations` on `product_category_name` to
add the English names. Store the result in `items_cat`.

Then compute **total revenue per English category** (sum of `price`, grouped by
`product_category_name_english`) and print the **top 5** categories.

**Expected output (rounded to 2 decimals):**
```
product_category_name_english
health_beauty            1258681.34
watches_gifts            1205005.68
bed_bath_table           1036988.68
sports_leisure            988048.97
computers_accessories     911954.32
Name: price, dtype: float64
```


In [20]:
from numpy.strings import title
from numpy._core.defchararray import upper
# Apply the 'upper' function to the 'product_category_name_english' column
product_translations['product_category_name_english'] = product_translations['product_category_name_english'].apply(title)

# Display the head of the modified DataFrame to see the changes
print(product_translations.head())

    product_category_name product_category_name_english
0            beleza_saude                 Health_Beauty
1  informatica_acessorios         Computers_Accessories
2              automotivo                          Auto
3         cama_mesa_banho                Bed_Bath_Table
4        moveis_decoracao               Furniture_Decor


In [21]:
items_cat = items_products.merge(
    product_translations,
    on='product_category_name',
    how='left'
)

top5_revenue = (
    items_cat.groupby('product_category_name_english')['price']
        .sum()
        .sort_values(ascending=False)
        .head(5)
)
print(top5_revenue)


product_category_name_english
Health_Beauty            1258681.34
Watches_Gifts            1205005.68
Bed_Bath_Table           1036988.68
Sports_Leisure            988048.97
Computers_Accessories     911954.32
Name: price, dtype: float64


---
## Group Exercise (25 min) — Build the full merged table

Work with your team. You'll chain **four** merges to assemble one complete
analysis table, then use it to answer two business questions.

### Question 6 — Assemble the full table

Starting from `orders`, chain these merges in order and store the result in `full`:

1. `.merge(customers, on='customer_id')` — inner
2. `.merge(order_items, on='order_id')` — inner
3. `.merge(products[['product_id', 'product_category_name']], on='product_id', how='left')`
4. `.merge(product_translations, on='product_category_name', how='left')`

Print `full.shape` and confirm it equals `(112650, 20)`.

**Expected output:**
```
(112650, 20)
Match: True
```


In [22]:
full = (
    orders
    .merge(customers, on='customer_id')  # inner join
    .merge(order_items, on='order_id')  # inner join
    .merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
    .merge(product_translations, on='product_category_name', how='left')
)

print(full.shape)
assert full.shape == (112650, 20), f"Expected (112650, 20), got {full.shape}"


(112650, 20)


### Question 7 — Total revenue from SP state

Using the `full` table, filter to rows where `customer_state == 'SP'` and sum the
`price` column. Print the total, rounded to 2 decimals.

**Expected output:**
```
5202955.05
```


In [23]:
sp_total = full.loc[full['customer_state'] == 'SP', 'price'].sum()
print(round(sp_total, 2))


5202955.05


### Question 8 — Top category by revenue in RJ state

Using the `full` table, filter to `customer_state == 'RJ'`, then find which
`product_category_name_english` has the highest total revenue (sum of `price`).
Print the category name and its revenue.

*Hint: filter first, then `groupby('product_category_name_english')['price'].sum()`
and use `.idxmax()` / `.max()` (or `.nlargest(1)`).*

**Expected output format** (your team discovers the actual category and value):
```
Top category in RJ: <category_name>  —  R$<revenue>
```


In [24]:
rj = full.loc[full['customer_state'] == 'RJ']
rj_by_cat = rj.groupby('product_category_name_english')['price'].sum()

top_cat = rj_by_cat.idxmax()
top_rev = rj_by_cat.max()

print(top_cat, round(top_rev, 2))

Watches_Gifts 185379.65
